In [1]:
!nvidia-smi
!nvcc --version | tail -1
import time
import numpy as np
import cupy as cp
print(cp.cuda.runtime.getDeviceProperties(0)["name"].decode(),
      "| CuPy", cp.__version__)

Mon Oct 12 03:43:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   27C    P0             47W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
def medir(fn, reps=5, sync=None):
    fn()  # calentamiento: compila kernels, reserva memoria
    if sync: sync()
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn()
        if sync: sync()
        ts.append(time.perf_counter() - t0)
    return np.mean(ts), np.std(ts)

sync = cp.cuda.Device().synchronize

for n in [256, 1024, 4096]:
    A  = np.random.rand(n, n).astype(np.float32)
    B  = np.random.rand(n, n).astype(np.float32)
    Ag, Bg = cp.asarray(A), cp.asarray(B)   # copia a GPU (fuera de la región medida)

    t_cpu, s_cpu = medir(lambda: A @ B)
    t_gpu, s_gpu = medir(lambda: Ag @ Bg, sync=sync)
    t_mal, _     = medir(lambda: Ag @ Bg)   # sin sync: mal medido a propósito

    print(f"n={n:5d}  CPU {t_cpu:.4f}±{s_cpu:.4f} s  "
          f"GPU {t_gpu:.5f}±{s_gpu:.5f} s  "
          f"speedup {t_cpu/t_gpu:7.1f}x  |  "
          f"GPU sin sync {t_mal:.6f} s (INCORRECTO)")

n=  256  CPU 0.0001±0.0000 s  GPU 0.00004±0.00004 s  speedup     1.6x  |  GPU sin sync 0.000011 s (INCORRECTO)
n= 1024  CPU 0.0015±0.0000 s  GPU 0.00007±0.00000 s  speedup    23.5x  |  GPU sin sync 0.000018 s (INCORRECTO)
n= 4096  CPU 0.0423±0.0003 s  GPU 0.00185±0.00002 s  speedup    22.8x  |  GPU sin sync 0.000014 s (INCORRECTO)
